# 🚗 Comparaison Multi-Architectures pour la Classification des Dommages Automobiles

Ce notebook présente une étude comparative entre trois architectures de pointe : **EfficientNet**, **ResNet**, et **Vision Transformer (ViT)**. L'objectif est d'identifier le modèle le plus performant pour le système EliteRent.

## 1. Travaux Connexes (Related Works)

L'évolution de la vision par ordinateur a mené à ces trois approches majeures :
- **ResNet (Residual Networks)** : Introduit par He et al., il résout le problème de disparition du gradient via des connexions de saut (skip connections).
- **EfficientNet** : Proposé par Tan & Le, il optimise simultanément la profondeur, la largeur et la résolution pour une efficacité maximale.
- **Vision Transformer (ViT)** : Adapté du NLP par Dosovitskiy et al., il utilise des mécanismes d'attention sur des patchs d'images, capturant des relations globales complexes.

---

In [ ]:
from roboflow import Roboflow
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix

# 1. Chargement du dataset
rf = Roboflow(api_key="FclaP7kXAJ9yTNCxwyCy")
project = rf.workspace("gaetano").project("car-damage-segmentation-zzed4")
version = project.version(4)
dataset = version.download("classification")
dataset_dir = dataset.location

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device : {device}")

In [ ]:
# 2. Prétraitement et Data Augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(os.path.join(dataset_dir, 'train'), transform=transform)
val_ds = datasets.ImageFolder(os.path.join(dataset_dir, 'valid'), transform=transform)
test_ds = datasets.ImageFolder(os.path.join(dataset_dir, 'test'), transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

num_classes = len(train_ds.classes)
class_names = train_ds.classes

In [ ]:
def train_model(model_name, num_epochs=5):
    print(f"\n--- Entraînement de {model_name} ---")
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        model.train()
        loop = tqdm(train_loader, leave=False)
        for imgs, labels in loop:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            loop.set_description(f"Epoch {epoch+1}/{num_epochs}")
            
        # Validation simple
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                _, pred = outputs.max(1)
                total += labels.size(0)
                correct += pred.eq(labels).sum().item()
        
        acc = correct / total
        history['val_acc'].append(acc)
        print(f"Epoch {epoch+1}: Val Acc = {acc:.4f}")
        
    return model, history

In [ ]:
# 3. Lancement de la comparaison
models_to_test = {
    'EfficientNet-B0': 'efficientnet_b0',
    'ResNet-50': 'resnet50',
    'Vision Transformer': 'vit_tiny_patch16_224'
}

results = {}
trained_models = {}

for name, model_id in models_to_test.items():
    m, hist = train_model(model_id, num_epochs=5)
    results[name] = hist['val_acc']
    trained_models[name] = m

In [ ]:
# 4. Visualisation des performances
plt.figure(figsize=(10, 6))
for name, acc_list in results.items():
    plt.plot(acc_list, label=name, marker='o')

plt.title("Comparaison de la Précision de Validation")
plt.xlabel("Époque")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 6. Analyse des Résultats et Choix du Gagnant

Après comparaison, nous sélectionnons le modèle ayant la meilleure accuracy sur l'ensemble de test pour le déploiement final.

In [ ]:
# 5. Sauvegarde du meilleur modèle
best_model_name = max(results, key=lambda k: results[k][-1])
print(f"🏆 Le gagnant est : {best_model_name}")

torch.save(trained_models[best_model_name].state_dict(), 'best_model.pth')
print(f"✅ Modèle '{best_model_name}' sauvegardé sous 'best_model.pth'")